In [ ]:
import requests
import json
import time

# Configuration
BASE_URL = "https://api.daricomma.com"
# REPLACE THIS TOKEN: Found in Browser -> Inspect -> Network -> Authorization Header
HEADERS = {
    "Authorization": "Bearer YOUR_ACCESS_TOKEN",
    "Content-Type": "application/json",
    "Accept": "application/json"
}

def collect_all_questions(total_pages=35):
    """
    Loops through all pages to remove the need for pagination 
    and returns a single consolidated list.
    """
    all_questions = []
    
    print(f"--- Starting Data Collection (Target: {total_pages} pages) ---")
    
    for page in range(1, total_pages + 1):
        # The 'field' or 'questions' endpoint usually takes a 'page' parameter
        endpoint = f"{BASE_URL}/field" 
        params = {"page": page}
        
        try:
            response = requests.get(endpoint, headers=HEADERS, params=params)
            
            if response.status_code == 200:
                data = response.json()
                
                # Daricomma typically wraps the list in a 'data' key
                questions_on_page = data.get('data', [])
                all_questions.extend(questions_on_page)
                
                print(f"Successfully collected Page {page}")
                
                # Small delay to prevent server-side rate limiting
                time.sleep(0.3) 
            else:
                print(f"Stopped at Page {page}. Status: {response.status_code}")
                break
                
        except Exception as e:
            print(f"An error occurred on page {page}: {e}")
            break

    print(f"\n--- Process Complete ---")
    print(f"Total items collected: {len(all_questions)}")
    return all_questions

if __name__ == "__main__":
    # 1. Fetch all data
    final_data_list = collect_all_questions(total_pages=35)

    # 2. Show all questions on "one page" (the console/terminal)
    for idx, item in enumerate(final_data_list, 1):
        # Adjust 'question_text' to the actual key used in the JSON response
        question = item.get('question_text', 'No text found')
        print(f"{idx}. {question}")

    # 3. Save to a single file for easy viewing
    with open("all_questions.json", "w", encoding="utf-8") as f:
        json.dump(final_data_list, f, ensure_ascii=False, indent=4)
    print("\nAll data saved to 'all_questions.json'")